# QKD BB84 Phase Control (Single Channel + Auto-Cycle)

Control notebook for the QKD phase-encoded BB84 system on the RFSoC 4x2.

## Register Map

| Address | Name | R/W | Description |
|---------|------|-----|-------------|
| 0x00 | CTRL | R/W | `[0]` global_en, `[1]` alice_en, `[2]` auto_cycle_en |
| 0x04 | ALICE_PHASE_STAGED | R/W | `[1:0]` Staged Alice phase |
| 0x08 | AUTO_FREQ_HZ | R | `[31:0]` Current auto-cycle frequency in Hz |
| 0x0C | STATUS | R | `[0]` alice_running, `[1]` auto_cycling, `[2]` sw_mode |
| 0x10 | PHASE_APPLY | R/W | Write 1 to latch staged phase (auto-clears) |
| 0x14 | ALICE_PHASE_ACTIVE | R | `[1:0]` Active Alice phase |
| 0x18 | AUTO_PERIOD_TICKS | R | `[31:0]` Auto-cycle period in RFDC clock ticks |
| 0x1C | VERSION | R | 0x2026_0505 |

## Phase Encoding

| Value | Phase | Basis | Bit |
|-------|-------|-------|-----|
| 0b00 | 0 | Z | 0 |
| 0b01 | pi/2 | X | 0 |
| 0b10 | pi | Z | 1 |
| 0b11 | 3pi/2 | X | 1 |

## Operating Modes

- **SW3=0**: Switch mode (SW0/SW1 set Alice phase directly)
- **SW3=1, SW2=0**: Register mode (AXI-Lite staged/apply)
- **SW3=1, SW2=1**: Auto-cycle mode (PL cycles through all 4 phases, buttons control rate)

## 1. Load Overlay and Initialize Hardware

In [17]:
from pynq import PL
PL.reset()

import xrfdc
import xrfclk
from pynq import Overlay, MMIO, Clocks
import pprint

# Load the bitstream — update path to match your build output
xrfclk.set_ref_clks()
ol = Overlay('./qkd_phase_bb84.bit')

# Show all IP in the design
pprint.pprint(ol.ip_dict)

RuntimeError: Frequency 122.88 MHz is not valid.

In [2]:
# Get handles to the QKD wrapper and RF data converter
# NOTE: update these names to match your block design instance names
# qkd = ol.qkd_top_wrapper_bd_0
qkd = ol.ip_dict['qkd_top_wrapper_bd_0']
base_addr = ol.ip_dict['qkd_top_wrapper_bd_0']['phys_addr']
addr_range = ol.ip_dict['qkd_top_wrapper_bd_0']['addr_range']
print(f"Base: {base_addr:#010x}, Range: {addr_range:#x}")
qkd_mmio = MMIO(base_addr, addr_range)
print(f"VERSION: {qkd_mmio.read(0x1C):#010x}")

rf = ol.usp_rf_data_converter_0
print(f"RF Data Converter: {rf}")

Base: 0x80000000, Range: 0x1000
VERSION: 0x20260425
RF Data Converter: <xrfdc.RFdc object at 0xffff7c152350>


## 2. Register Access Helpers

In [ ]:
# Register offsets
REG_CTRL                = 0x00
REG_ALICE_PHASE_STAGED  = 0x04
REG_AUTO_FREQ_HZ        = 0x08
REG_STATUS              = 0x0C
REG_PHASE_APPLY         = 0x10
REG_ALICE_PHASE_ACTIVE  = 0x14
REG_AUTO_PERIOD_TICKS   = 0x18
REG_VERSION             = 0x1C

RFDC_CLK_FREQ_HZ = 307_200_000  # 8x interpolation: 4915.2 / (8 * 2) = 307.2 MHz

PHASE_LABELS = {0: '0', 1: 'pi/2', 2: 'pi', 3: '3pi/2'}

def reg_read(offset):
    return qkd_mmio.read(offset)

def reg_write(offset, value):
    qkd_mmio.write(offset, value)

def set_phase(alice_phase):
    """Stage and apply Alice phase.
    
    Args:
        alice_phase: 0-3 (0=0, 1=pi/2, 2=pi, 3=3pi/2)
    """
    reg_write(REG_ALICE_PHASE_STAGED, alice_phase & 0x3)
    reg_write(REG_PHASE_APPLY, 1)

def get_status():
    """Read and decode the STATUS register."""
    s = reg_read(REG_STATUS)
    return {
        'alice_running': bool(s & 0x1),
        'auto_cycling':  bool(s & 0x2),
        'sw_mode':       bool(s & 0x4),
    }

def get_active_phase():
    """Read back the currently active Alice phase."""
    a = reg_read(REG_ALICE_PHASE_ACTIVE) & 0x3
    return PHASE_LABELS[a]

def get_auto_freq():
    """Read current auto-cycle frequency in Hz."""
    return reg_read(REG_AUTO_FREQ_HZ)

def get_auto_period():
    """Read current auto-cycle period in RFDC ticks."""
    return reg_read(REG_AUTO_PERIOD_TICKS)

def enable_auto_cycle():
    """Enable auto-cycle mode via CTRL register (also needs SW2=1 or this bit)."""
    ctrl = reg_read(REG_CTRL)
    reg_write(REG_CTRL, ctrl | 0x04)  # set auto_cycle_en

def disable_auto_cycle():
    """Disable auto-cycle mode via CTRL register."""
    ctrl = reg_read(REG_CTRL)
    reg_write(REG_CTRL, ctrl & ~0x04)  # clear auto_cycle_en

# Verify connectivity
version = reg_read(REG_VERSION)
print(f"VERSION: {version:#010x}")
assert version == 0x2026_0505, f"Unexpected version: {version:#010x}"

## 3. Configure RF Data Converter

DAC tile 0, slice 0 (Alice). NCO at 150 MHz for AOM, I/Q -> Real mixer mode.

In [ ]:
dac_tile = rf.dac_tiles[0]
print(f"DAC Tile 0: {dac_tile}; PLL locked = {dac_tile.PLLLockStatus}")

block = dac_tile.blocks[0]
print(f"\nAlice (slice 0):")
print(f"  Status: {block.BlockStatus}")
print(f"  Mixer:  {block.MixerSettings}")

# Set NCO to 150 MHz for AOM
block.MixerSettings['Freq'] = 150.0
block.UpdateEvent(xrfdc.EVENT_MIXER)
print(f"  Mixer (after update): {block.MixerSettings}")

## 4. Enable Outputs

Make sure SW3 is in register mode (SW3=1) on the board before running this cell.

In [ ]:
# Enable global + Alice (no auto-cycle yet — use SW2 or enable_auto_cycle() for that)
reg_write(REG_CTRL, 0x03)

status = get_status()
print(f"Status: {status}")
assert status['alice_running'], "Alice not running — check CTRL and SW3"

## 5. Set Phase (Register Mode)

Use `set_phase(value)` to update Alice phase. Requires SW3=1, SW2=0.

Phase encoding: `0` = 0, `1` = pi/2, `2` = pi, `3` = 3pi/2

In [ ]:
set_phase(0)
print(f"Active phase: {get_active_phase()}")

# Cycle through all 4 phases
import time
for p in range(4):
    set_phase(p)
    time.sleep(0.1)
    print(f"Phase {p} ({PHASE_LABELS[p]}): active = {get_active_phase()}")

## 6. Auto-Cycle Mode (AOM Bandwidth Test)

Enable auto-cycle to have the PL automatically sweep through all 4 phases.
Use the board buttons to control switching rate:
- **btn[3]**: Cycle unit (Hz / KHz / MHz)
- **btn[2]**: Cycle increment (1 / 5 / 10)
- **btn[1]**: Increase frequency
- **btn[0]**: Decrease frequency

LED[3] lights up when auto-cycle is active. LED[1:0] shows current phase.

In [ ]:
# Enable auto-cycle (can also flip SW2=1 on the board for hardware-only operation)
enable_auto_cycle()

status = get_status()
print(f"Status: {status}")
print(f"Auto-cycle freq: {get_auto_freq()} Hz")
period = get_auto_period()
if period > 0:
    computed_freq = RFDC_CLK_FREQ_HZ / period
    print(f"Auto-cycle period: {period} ticks ({computed_freq:.1f} Hz)")
print("\nUse buttons on the board to adjust switching speed.")
print("Watch the scope to find the AOM bandwidth limit.")

## 7. Monitor Auto-Cycle (optional)

Poll and display the current switching frequency while you adjust with buttons.

In [ ]:
import time
from IPython.display import clear_output

try:
    for _ in range(30):  # monitor for 30 seconds
        freq = get_auto_freq()
        period = get_auto_period()
        phase = get_active_phase()
        clear_output(wait=True)
        print(f"Auto-cycle frequency: {freq} Hz")
        if period > 0:
            print(f"Period: {period} ticks ({RFDC_CLK_FREQ_HZ/period:.1f} Hz actual)")
        print(f"Current phase: {phase}")
        print(f"Status: {get_status()}")
        print("\n[Ctrl+C to stop monitoring]")
        time.sleep(1.0)
except KeyboardInterrupt:
    print("Monitoring stopped.")

In [ ]:
# Disable everything
disable_auto_cycle()
reg_write(REG_CTRL, 0x00)
print(f"Status: {get_status()}")